# Reference
- https://github.com/MyoHub/myosuite/blob/main/docs/source/tutorials/4c_Train_SB_policy.ipynb

# Common Libraries

In [1]:
import sys
import os
os.environ["MUJOCO_GL"] = "egl"
import skvideo.io
import numpy as np
from myosuite.utils import gym

MyoSuite:> Registering Myo Envs


/home/seojin/anaconda3/envs/DP/lib/python3.10/site-packages/gymnasium/envs/registration.py:694: UserWarning: WARN: Overriding environment myoArmReachFixed-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/home/seojin/anaconda3/envs/DP/lib/python3.10/site-packages/gymnasium/envs/registration.py:694: UserWarning: WARN: Overriding environment myoSarcArmReachFixed-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/home/seojin/anaconda3/envs/DP/lib/python3.10/site-packages/gymnasium/envs/registration.py:694: UserWarning: WARN: Overriding environment myoFatiArmReachFixed-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")
/home/seojin/anaconda3/envs/DP/lib/python3.10/site-packages/gymnasium/envs/registration.py:694: UserWarning: WARN: Overriding environment myoArmReachRandom-v0 already in registry.
  logger.warn(f"Overriding environment {new_spe

# Custom Libraries

In [2]:
sys.path.append("/home/seojin/Seojin_commonTool/Module")
from jupyter_util import show_video

# Environment

In [3]:
env = gym.make('myoElbowPose1D6MRandom-v0')

env.reset();

    MyoSuite: A contact-rich simulation suite for musculoskeletal motor control
        Vittorio Caggiano, Huawei Wang, Guillaume Durandau, Massimo Sartori, Vikash Kumar
        L4DC-2019 | https://sites.google.com/view/myosuite
    


In [4]:
from stable_baselines3 import PPO

model = PPO("MlpPolicy", env, verbose=0)

print("========================================")
print("Starting policy learning")
print("========================================")

model.learn(total_timesteps=1000)

print("========================================")
print("Job Finished.")
print("========================================")

model.save('ElbowPose_policy')


2026-03-27 18:58:00.743484: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774605480.761173 3129516 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774605480.766590 3129516 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774605480.781362 3129516 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774605480.781376 3129516 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774605480.781378 3129516 computation_placer.cc:177] computation placer alr

Starting policy learning
Job Finished.


In [5]:
policy = "ElbowPose_policy.zip"

pi = PPO.load(policy)

# define a discrete sequence of positions to test
AngleSequence = [60, 30, 30, 60, 80, 80, 60, 30, 80, 30, 80, 60]
env.reset()
frames = []
for ep in range(len(AngleSequence)):
    print("Ep {} of {} testing angle {}".format(ep, len(AngleSequence), AngleSequence[ep]))
    env.unwrapped.target_jnt_value = [np.deg2rad(AngleSequence[int(ep)])]
    env.unwrapped.target_type = 'fixed'
    env.unwrapped.weight_range=(0,0)
    env.unwrapped.update_target()
    for _ in range(40):
        frame = env.unwrapped.sim.renderer.render_offscreen(
                        width=400,
                        height=400,
                        camera_id=0)
        frames.append(frame)
        o = env.unwrapped.get_obs()
        a = pi.predict(o)[0]
        next_o, r, done, *_, ifo = env.step(a) # take an action based on the current observation
env.close()

os.makedirs('videos', exist_ok=True)
# make a local copy
skvideo.io.vwrite('videos/arm.mp4', np.asarray(frames),outputdict={"-pix_fmt": "yuv420p"})
show_video('videos/arm.mp4')

Ep 0 of 12 testing angle 60
Ep 1 of 12 testing angle 30
Ep 2 of 12 testing angle 30
Ep 3 of 12 testing angle 60
Ep 4 of 12 testing angle 80
Ep 5 of 12 testing angle 80
Ep 6 of 12 testing angle 60
Ep 7 of 12 testing angle 30
Ep 8 of 12 testing angle 80
Ep 9 of 12 testing angle 30
Ep 10 of 12 testing angle 80
Ep 11 of 12 testing angle 60


In [6]:
env.unwrapped.update_target

<bound method PoseEnvV0.update_target of <myosuite.envs.myo.myobase.pose_v0.PoseEnvV0 object at 0x70cfb3960e80>>